In [2]:
import pandas as pd 

orders = pd.read_csv("../data/raw/olist_orders_dataset.csv")
orders_items = pd.read_csv("../data/raw/olist_order_items_dataset.csv")
sellers = pd.read_csv("../data/raw/olist_sellers_dataset.csv")
order_reviews = pd.read_csv("../data/raw/olist_order_reviews_dataset.csv")

In [3]:
orders_delivered = orders [
    (orders["order_status"] == 'delivered')
].copy()

len(orders_delivered)

96478

In [4]:
total_sellers = orders_delivered.merge(
    orders_items[["order_id", "seller_id", "price"]],
    on="order_id",
    how="left"
)
  
len(total_sellers)

110197

In [5]:
total_per_sellers = total_sellers.groupby("seller_id") ["price"].sum()

total_per_sellers.sort_values(ascending=False).head(5)

seller_id
4869f7a5dfa277a7dca6462dcf3b52b2    226987.93
53243585a1d6dc2643021fd1853d8905    217940.44
4a3ca9315b744ce9f8e9374361493884    196882.12
fa1c13f2614d7b5c4749cbc52fecda94    190917.14
7c67e1448b00f6e969d365cea6b010ab    186570.05
Name: price, dtype: float64

In [6]:
pareto_df = total_per_sellers.sort_values(ascending=False).reset_index() 
pareto_df.columns = ["seller_id", "revenue"]

pareto_df.head()

,seller_id,revenue
0,4869f7a5dfa277a7dca6462dcf3b52b2,226987.93
1,53243585a1d6dc2643021fd1853d8905,217940.44
2,4a3ca9315b744ce9f8e9374361493884,196882.12
3,fa1c13f2614d7b5c4749cbc52fecda94,190917.14
4,7c67e1448b00f6e969d365cea6b010ab,186570.05


In [7]:
pareto_df["revenue_accum"] = pareto_df["revenue"].cumsum()

pareto_df["pct_revenue"] = (pareto_df["revenue_accum"] / pareto_df["revenue"].sum()) * 100

pareto_df.head()

,seller_id,revenue,revenue_accum,pct_revenue
0,4869f7a5dfa277a7dca6462dcf3b52b2,226987.93,226987.93,1.716809
1,53243585a1d6dc2643021fd1853d8905,217940.44,444928.37,3.365189
2,4a3ca9315b744ce9f8e9374361493884,196882.12,641810.49,4.854295
3,fa1c13f2614d7b5c4749cbc52fecda94,190917.14,832727.63,6.298285
4,7c67e1448b00f6e969d365cea6b010ab,186570.05,1019297.68,7.709396


In [8]:
pareto_df[pareto_df["pct_revenue"] >= 80].head(1)

,seller_id,revenue,revenue_accum,pct_revenue
532,e06f09ec0a4aca210779cf1cfc63cf19,5279.49,10579610.39,80.018242


In [9]:
top_sellers = total_sellers.groupby("seller_id").agg(
    items_sold=("order_id", "count"),
    revenue=("price", "sum")
).sort_values("revenue", ascending=False).head(10).reset_index()

In [10]:
top_sellers = top_sellers.merge(
    sellers[["seller_id", "seller_state"]],
    on="seller_id",
    how="left"
)
top_sellers

,seller_id,items_sold,revenue,seller_state
0,4869f7a5dfa277a7dca6462dcf3b52b2,1148,226987.93,SP
1,53243585a1d6dc2643021fd1853d8905,400,217940.44,BA
2,4a3ca9315b744ce9f8e9374361493884,1949,196882.12,SP
3,fa1c13f2614d7b5c4749cbc52fecda94,579,190917.14,SP
4,7c67e1448b00f6e969d365cea6b010ab,1355,186570.05,SP
5,7e93a43ef30c4f03f38b393420bc753a,322,165981.49,SP
6,da8622b14eb17ae2831f4ac5b9dab84a,1548,159816.87,SP
7,7a67c85e85bb2ce8582c35f2203ad736,1155,139658.69,SP
8,1025f0e2d44d7041d6cf58b6550e0bfa,1420,138208.56,SP
9,955fee9216a65b617aa5c0531780ce60,1472,131836.71,SP


In [11]:
top_orders = total_sellers[total_sellers["seller_id"].isin(top_sellers["seller_id"])]

top_orders_reviews = top_orders.merge(
    order_reviews[["order_id", "review_score"]],
    on="order_id",
    how="left"
)

top_seller_score = total_sellers.merge(
    order_reviews[["order_id", "review_score"]],
    on="order_id",
    how="left"
)

top_seller_score = top_orders_reviews.groupby("seller_id") ["review_score"].mean().round(2).reset_index()
top_seller_score.columns = ["seller_id", "avg_review_score"]

top_sellers = top_sellers.merge(top_seller_score, on="seller_id", how="left")
top_sellers

,seller_id,items_sold,revenue,seller_state,avg_review_score
0,4869f7a5dfa277a7dca6462dcf3b52b2,1148,226987.93,SP,4.14
1,53243585a1d6dc2643021fd1853d8905,400,217940.44,BA,4.13
2,4a3ca9315b744ce9f8e9374361493884,1949,196882.12,SP,3.83
3,fa1c13f2614d7b5c4749cbc52fecda94,579,190917.14,SP,4.37
4,7c67e1448b00f6e969d365cea6b010ab,1355,186570.05,SP,3.35
5,7e93a43ef30c4f03f38b393420bc753a,322,165981.49,SP,4.36
6,da8622b14eb17ae2831f4ac5b9dab84a,1548,159816.87,SP,4.08
7,7a67c85e85bb2ce8582c35f2203ad736,1155,139658.69,SP,4.27
8,1025f0e2d44d7041d6cf58b6550e0bfa,1420,138208.56,SP,3.87
9,955fee9216a65b617aa5c0531780ce60,1472,131836.71,SP,4.09
